# 30 — Agent Evaluation Dataset

## 1. Imports

In [1]:
import os
import json
import dotenv
import openai
from langsmith import Client

dotenv.load_dotenv('../.env')

ls_client = Client(api_key=os.getenv('LANGSMITH_API_KEY'))
print('✓ LangSmith client ready')

✓ LangSmith client ready


## 2. Routing Examples

In [2]:
routing_examples = [
    # --- Route to troubleshooting ---
    {
        "inputs": {
            "user_message": "HX-200 high oil temperature above 80°C. What could be causing this?"
        },
        "outputs": {
            "expected_route": "troubleshooting",
            "expected_agent_behavior": "Start RCA investigation: check machine existence, retrieve procedure context for high oil temperature, search CM history for similar events on HX-200, build hypothesis table.",
            "ground_truth_answer": "Route to troubleshooting agent for RCA on HX-200 high oil temperature."
        },
        "metadata": {"scenario": "routing_troubleshooting", "machine": "HX-200", "fault_code": "E-001"}
    },
    {
        "inputs": {
            "user_message": "CB-200 belt is slipping and the motor current is spiking. Help me diagnose."
        },
        "outputs": {
            "expected_route": "troubleshooting",
            "expected_agent_behavior": "Start RCA on CB-200 belt slip / motor current spike. Check known issue categories, retrieve procedure context for belt tension and motor overload, search CM history.",
            "ground_truth_answer": "Route to troubleshooting agent for RCA on CB-200 belt slip."
        },
        "metadata": {"scenario": "routing_troubleshooting", "machine": "CB-200", "fault_code": "B-002"}
    },
    {
        "inputs": {
            "user_message": "IH-300 fault H-001 just triggered — coil temperature alarm."
        },
        "outputs": {
            "expected_route": "troubleshooting",
            "expected_agent_behavior": "Start RCA on IH-300 coil temperature alarm H-001. Retrieve procedure for coil cooling fault, query CM history and knowledge graph for recurrence patterns.",
            "ground_truth_answer": "Route to troubleshooting agent for RCA on IH-300 coil overtemperature."
        },
        "metadata": {"scenario": "routing_troubleshooting", "machine": "IH-300", "fault_code": "H-001"}
    },
    {
        "inputs": {
            "user_message": "CNC-500 spindle vibration is unusually high. Machine tripping during cycle."
        },
        "outputs": {
            "expected_route": "troubleshooting",
            "expected_agent_behavior": "Start RCA on CNC-500 high spindle vibration. Retrieve procedure context for spindle bearing wear and vibration faults, check CM history and sensor anomaly summary.",
            "ground_truth_answer": "Route to troubleshooting agent for RCA on CNC-500 spindle vibration."
        },
        "metadata": {"scenario": "routing_troubleshooting", "machine": "CNC-500", "fault_code": "C-005"}
    },

    # --- Route to summarizer ---
    {
        "inputs": {
            "user_message": "Show me all B-002 bearing failure interventions on CB-200 in 2025."
        },
        "outputs": {
            "expected_route": "summarizer",
            "expected_agent_behavior": "List intervention IDs for CB-200 in 2025, filter for B-002 bearing-related faults, summarize each, append to summaries.md.",
            "ground_truth_answer": "Route to summarizer agent to retrieve and summarize CB-200 B-002 bearing failure interventions from 2025."
        },
        "metadata": {"scenario": "routing_summarizer", "machine": "CB-200", "fault_code": "B-002"}
    },
    {
        "inputs": {
            "user_message": "Build a known-case template for high oil temperature faults on HX machines."
        },
        "outputs": {
            "expected_route": "summarizer",
            "expected_agent_behavior": "Retrieve existing templates, list relevant interventions for HX machines with high oil temp, summarize each, build template, validate, and save.",
            "ground_truth_answer": "Route to summarizer agent to build a known-case template for HX high oil temperature faults."
        },
        "metadata": {"scenario": "routing_summarizer", "machine": "HX", "fault_code": "E-001"}
    },
    {
        "inputs": {
            "user_message": "Has this kind of lubrication flow fault happened before on CR-100?"
        },
        "outputs": {
            "expected_route": "summarizer",
            "expected_agent_behavior": "Search CM history for lubrication-related interventions on CR-100, summarize findings.",
            "ground_truth_answer": "Route to summarizer agent to search and summarize past lubrication fault interventions on CR-100."
        },
        "metadata": {"scenario": "routing_summarizer", "machine": "CR-100", "fault_code": None}
    },

    # --- Direct coordinator answer ---
    {
        "inputs": {
            "user_message": "What is the weather like today?"
        },
        "outputs": {
            "expected_route": "direct",
            "expected_agent_behavior": "Coordinator answers directly without routing. Informs user this is a maintenance assistant.",
            "ground_truth_answer": "I'm a maintenance assistant — I can help with machine diagnoses or past intervention summaries."
        },
        "metadata": {"scenario": "routing_direct", "machine": None, "fault_code": None}
    },
    {
        "inputs": {
            "user_message": "Hi!"
        },
        "outputs": {
            "expected_route": "direct",
            "expected_agent_behavior": "Coordinator responds with the standard greeting/capability overview — no routing.",
            "ground_truth_answer": "Hello, I'm Hephaestus, your maintenance assistant. I can help you with root cause analysis, procedure lookup, historical interventions, and troubleshooting guidance."
        },
        "metadata": {"scenario": "routing_direct", "machine": None, "fault_code": None}
    },
    {
        "inputs": {
            "user_message": "Something is wrong with a machine."
        },
        "outputs": {
            "expected_route": "direct",
            "expected_agent_behavior": "Coordinator asks for clarification (machine ID + symptom) rather than routing.",
            "ground_truth_answer": "Could you tell me which machine is affected and what symptom you're seeing?"
        },
        "metadata": {"scenario": "routing_direct", "machine": None, "fault_code": None}
    },
]

print(f'Routing examples: {len(routing_examples)}')

Routing examples: 10


## 3. Troubleshooting Examples

In [ ]:
EVAL_DIRECTIVE = (
    " [SINGLE-TURN EVAL] This is an automated evaluation. Today is 2025-05-01. "
    "Run all investigation steps (check_machine_exists, procedures, CM history, knowledge graph) "
    "then go directly to a recommendation with a hypothesis table and action plan. "
    "Do NOT ask discrimination questions or request technician confirmation."
)

troubleshooting_examples = [
    # --- Interventions only ---
    {
        "inputs": {
            "user_message": "HX-200 fault E-007, low coolant pressure. Machine halted." + EVAL_DIRECTIVE,
            "target_agent": "troubleshooting",
        },
        "outputs": {
            "expected_route": "troubleshooting",
            "expected_agent_behavior": "Retrieve CM history for HX-200 E-007 low coolant pressure events. Identify recurrence pattern and most common root cause from past interventions. Build hypothesis table citing INT-IDs. Deliver a final recommendation without asking follow-up questions.",
            "ground_truth_answer": "E-007 on HX-200 is typically caused by a blocked coolant filter or a failing coolant pump. Recommended actions: (1) check and clean or replace the coolant filter, (2) inspect coolant pump outlet pressure, (3) verify coolant reservoir level."
        },
        "metadata": {
            "scenario": "troubleshooting_interventions_only",
            "machine": "HX-200", "fault_code": "E-007",
            "expected_sources": ["cm_history"]
        }
    },
    {
        "inputs": {
            "user_message": "IH-300 coil temperature alarm H-001 keeps recurring." + EVAL_DIRECTIVE,
            "target_agent": "troubleshooting",
        },
        "outputs": {
            "expected_route": "troubleshooting",
            "expected_agent_behavior": "Query CM history for recurring H-001 on IH-300. Use fleet impact to check if other IH machines share the pattern. Deliver a final recommendation without asking follow-up questions.",
            "ground_truth_answer": "H-001 on IH-300 is a recurring fault seen in multiple 2025 interventions. Root causes vary between partial coolant flow restriction and coil insulation degradation. Preventive action: periodic coolant flow verification and insulation resistance checks."
        },
        "metadata": {
            "scenario": "troubleshooting_interventions_only",
            "machine": "IH-300", "fault_code": "H-001",
            "expected_sources": ["cm_history", "knowledge_graph"]
        }
    },

    # --- Procedure + interventions ---
    {
        "inputs": {
            "user_message": "HX-350 oil pressure dropped below 60 bar. Fault E-001." + EVAL_DIRECTIVE,
            "target_agent": "troubleshooting",
        },
        "outputs": {
            "expected_route": "troubleshooting",
            "expected_agent_behavior": "Retrieve procedure for low oil pressure on HX machines. Cross-reference with CM history for E-001 on HX-350. Cite both procedure steps and INT-IDs in hypothesis table. Deliver a final recommendation without asking follow-up questions.",
            "ground_truth_answer": "Low oil pressure on HX-350 (E-001) is commonly caused by pump coupling failure or seal wear. Procedure: (1) isolate and depressurise, (2) inspect coupling and seals, (3) replace worn parts, (4) verify pressure after restart. Past interventions confirm these root causes."
        },
        "metadata": {
            "scenario": "troubleshooting_procedure_intervention",
            "machine": "HX-350", "fault_code": "E-001",
            "expected_sources": ["procedure", "cm_history"]
        }
    },
    {
        "inputs": {
            "user_message": "CNC-500 fault C-005 — coolant contamination alarm mid-cycle." + EVAL_DIRECTIVE,
            "target_agent": "troubleshooting",
        },
        "outputs": {
            "expected_route": "troubleshooting",
            "expected_agent_behavior": "Retrieve procedure for coolant contamination on CNC machines. Check CM history for C-005 on CNC-500. Build decision-tree recommendation citing both sources. Deliver a final recommendation without asking follow-up questions.",
            "ground_truth_answer": "C-005 coolant contamination on CNC-500 is typically caused by oil ingress from spindle seals or bacterial growth. Procedure: (1) drain and flush tank, (2) inspect spindle seals, (3) refill with fresh coolant at correct concentration. CM history confirms spindle seal as dominant root cause."
        },
        "metadata": {
            "scenario": "troubleshooting_procedure_intervention",
            "machine": "CNC-500", "fault_code": "C-005",
            "expected_sources": ["procedure", "cm_history"]
        }
    },

    # --- Procedure only ---
    {
        "inputs": {
            "user_message": "HX-350 secondary pump has unusual vibration — never seen this before." + EVAL_DIRECTIVE,
            "target_agent": "troubleshooting",
        },
        "outputs": {
            "expected_route": "troubleshooting",
            "expected_agent_behavior": "When CM history returns sparse results, fall back to procedure context for pump vibration on HX machines. Build hypothesis table using procedure references. Acknowledge limited CM history. Deliver a final recommendation without asking follow-up questions.",
            "ground_truth_answer": "Limited CM history for secondary pump vibration on HX-350. Based on procedure: likely causes are pump bearing wear, impeller imbalance, or cavitation. Checks: (1) measure vibration signature, (2) inspect bearing clearance, (3) verify inlet pressure above cavitation threshold."
        },
        "metadata": {
            "scenario": "troubleshooting_procedure_only",
            "machine": "HX-350", "fault_code": None,
            "expected_sources": ["procedure"]
        }
    },
]

print(f'Troubleshooting examples: {len(troubleshooting_examples)}')

## 4. Summarizer Examples

In [ ]:
SUMMARIZER_DIRECTIVE = (
    " [SINGLE-TURN EVAL] The INT-IDs are already identified — go directly to Step 3 (summarize each INT-ID). "
    "Date window: 2025-04-01 to 2025-04-15. "
    "Do NOT ask for clarification, do NOT collect context, do NOT call list_intervention_ids_by_date. "
    "Call summarize_intervention for each ID and present all summaries."
)

summarizer_examples = [
    {
        "inputs": {
            "user_message": (
                "Summarize interventions INT-2025-0225 and INT-2025-0110 for CB-200 B-002 bearing fault."
                + SUMMARIZER_DIRECTIVE
            ),
            "target_agent": "summarizer",
        },
        "outputs": {
            "expected_route": "summarizer",
            "expected_agent_behavior": "Call summarize_intervention for INT-2025-0225 and INT-2025-0110 directly. Present both summaries with [INT: ID] markers. Do not ask clarifying questions.",
            "ground_truth_answer": "Both B-002 interventions on CB-200 should be summarized with: fault description, root cause, actions taken, and outcome. Each summary preserves the [INT: ID] marker.",
            "expected_int_ids": ["INT-2025-0225", "INT-2025-0110"],
            "expected_summaries_count": 2
        },
        "metadata": {"scenario": "summarizer_summaries", "machine": "CB-200", "fault_code": "B-002"}
    },
    {
        "inputs": {
            "user_message": (
                "Summarize H-001 coil temperature alarm interventions on IH-300 between 2025-01-01 and 2025-06-30. "
                "The relevant INT-IDs are: INT-2025-0042, INT-2025-0362, INT-2025-0545."
                + SUMMARIZER_DIRECTIVE
            ),
            "target_agent": "summarizer",
        },
        "outputs": {
            "expected_route": "summarizer",
            "expected_agent_behavior": "Call summarize_intervention for INT-2025-0042, INT-2025-0362, INT-2025-0545 directly. Present all three summaries with [INT: ID] markers. Do not ask clarifying questions.",
            "ground_truth_answer": "Summaries for H-001 interventions on IH-300 in 2025 should cover INT-2025-0042, INT-2025-0362, INT-2025-0545, each with: fault description, root cause, corrective actions, outcome.",
            "expected_int_ids": ["INT-2025-0042", "INT-2025-0362", "INT-2025-0545"],
            "expected_summaries_count": 3
        },
        "metadata": {"scenario": "summarizer_summaries", "machine": "IH-300", "fault_code": "H-001"}
    },
]

print(f'Summarizer examples: {len(summarizer_examples)}')

## 5. Combine All Examples

In [5]:
all_examples = routing_examples + troubleshooting_examples + summarizer_examples

# Count by scenario
from collections import Counter
scenario_counts = Counter(e['metadata']['scenario'] for e in all_examples)
print(f'Total examples: {len(all_examples)}')
for scenario, count in sorted(scenario_counts.items()):
    print(f'  {scenario}: {count}')

Total examples: 17
  routing_direct: 3
  routing_summarizer: 3
  routing_troubleshooting: 4
  summarizer_summaries: 2
  troubleshooting_interventions_only: 2
  troubleshooting_procedure_intervention: 2
  troubleshooting_procedure_only: 1


## 6. Upload to LangSmith

In [6]:
DATASET_NAME = 'hephaestus-agent-eval'

# Recreate dataset from scratch so stale examples are replaced
try:
    ls_client.delete_dataset(dataset_name=DATASET_NAME)
    print(f'✓ Deleted old dataset: {DATASET_NAME}')
except Exception:
    pass

dataset = ls_client.create_dataset(
    dataset_name=DATASET_NAME,
    description='Agent evaluation dataset for Hephaestus multi-agent system. '
                'Covers coordinator routing, troubleshooting RCA quality, and summarizer output completeness.'
)
print(f'✓ Created dataset: {DATASET_NAME} (id={dataset.id})')

✓ Deleted old dataset: hephaestus-agent-eval
✓ Created dataset: hephaestus-agent-eval (id=0c000daf-c9d6-43dd-9367-febd975b4077)


In [7]:
ls_examples = [
    {
        'inputs':   ex['inputs'],
        'outputs':  ex['outputs'],
        'metadata': ex['metadata'],
    }
    for ex in all_examples
]

ls_client.create_examples(dataset_id=dataset.id, examples=ls_examples)

print(f'\nUploaded {len(ls_examples)} examples to "{DATASET_NAME}"')
for scenario, count in sorted(scenario_counts.items()):
    print(f'  {scenario}: {count}')


Uploaded 17 examples to "hephaestus-agent-eval"
  routing_direct: 3
  routing_summarizer: 3
  routing_troubleshooting: 4
  summarizer_summaries: 2
  troubleshooting_interventions_only: 2
  troubleshooting_procedure_intervention: 2
  troubleshooting_procedure_only: 1


## 7. Verify Upload

In [8]:
uploaded = list(ls_client.list_examples(dataset_name=DATASET_NAME))
print(f'Examples in LangSmith: {len(uploaded)}')

scenarios = Counter(e.metadata.get('scenario') for e in uploaded)
print('\nBy scenario:')
for s, c in sorted(scenarios.items()):
    print(f'  {s}: {c}')

Examples in LangSmith: 17

By scenario:
  routing_direct: 3
  routing_summarizer: 3
  routing_troubleshooting: 4
  summarizer_summaries: 2
  troubleshooting_interventions_only: 2
  troubleshooting_procedure_intervention: 2
  troubleshooting_procedure_only: 1
